<a href="https://colab.research.google.com/github/sivabharathi-k/sivabharathi-codebooster-2026/blob/main/Day4/Day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Pyspark
!pip install pyspark
print('pyspark installed')



pyspark installed


In [ ]:
#import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year, month,to_date,col, round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
spark=SparkSession.builder \
    .appName('Day4_Bigdata_sales') \
    .config('spark.sql.adapative.enabled','true') \
    .getOrCreate()
print(f'Spark Version:{spark.version}')
print(f'SparkSession:Active')
print(f'application:{spark.sparkContext.appName}')

Spark Version:4.0.2
SparkSession:Active
application:Day4_Bigdata_sales


In [ ]:
#Load csv into pyspark Dataframe
df_bronze=spark.read \
  .option('header','true') \
  .option('inferSchema','true') \
  .csv('large_sales_data.csv')
print('=== BRONZE LAYER - Raw Data ===')
print(f'Rows:{df_bronze.count()}')
print(f'Columns:{len(df_bronze.columns)}')
df_bronze.printSchema()


=== BRONZE LAYER - Raw Data ===
Rows:5000
Columns:13
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [ ]:
print('first 5 rows')
df_bronze.show(5,truncate=False)

print('/nBasic statistics for numberc columns')
df_bronze.select('quantity','unit_price','revenue').describe().show()

first 5 rows
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Kum

In [ ]:
df_bronze.write \
  .mode('overwrite') \
  .parquet('sales_broze.parquet')
print('Bronze parquet saved: sales_bronze.parquet')

Bronze parquet saved: sales_bronze.parquet


In [ ]:
from os.path import isfile
df_bronze.write \
     .mode('overwrite')\
     .parquet('sales_bronze.parquet')
print('Bronze Parquet saved: sales_bronze.parquet')

import os
def get_dir_size(path):
    if os.path.isfile(path):
        return os.path.getsize(path) / 1024
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total / 1024

csv_size       = get_dir_size('large_sales_data.csv')
parquet_size   = get_dir_size('sales_bronze.parquet')
reduction      =(1 - parquet_size/csv_size) * 100

print(f'\nCSV Size: {csv_size:.2f} KB')
print(f'Parquet Size: {parquet_size:.2f} KB')
print(f'Reduction: {reduction:.1f}% smaller')
print(f'\nAT 1 TB scale: CSV=1000  GB -> Parquet={1000*(1-reduction/100):.0f} GB')

Bronze Parquet saved: sales_bronze.parquet

CSV Size: 529.31 KB
Parquet Size: 55.10 KB
Reduction: 89.6% smaller

AT 1 TB scale: CSV=1000  GB -> Parquet=104 GB


In [ ]:
df_bronze = df_bronze.withColumn('total_price_and_revenue', col('unit_price') + col('revenue'))
df_bronze.show(5, truncate=False)

df_bronze.printSchema()

df_bronze.select('unit_price','revenue','total_price_and_revenue').describe().show()

+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+-----------------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|total_price_and_revenue|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+-----------------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |286000                 |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |132000                 |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |8800 

In [ ]:
df_bronze.select('product','revenue').show(5)
df_bronze.filter(F.col("revenue")>50000).show()

df_bronze.groupby("category") \
  .agg(F.sum("revenue").alias("total_revenue")) \
  .orderBy(F.col("total_revenue"))

+----------+-------+
|   product|revenue|
+----------+-------+
|   Monitor| 264000|
|   Printer| 120000|
|     Mouse|   8000|
|    Tablet| 160000|
|Headphones|  14000|
+----------+-------+
only showing top 5 rows
+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+-----------------------+
|order_id|customer_name|product|   category|quantity|unit_price|revenue|order_date|     city|region|   sales_rep|  payment_method|order_status|total_price_and_revenue|
+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+-----------------------+
|    1001|  Sneha Reddy|Monitor|Electronics|      12|     22000| 264000|2023-05-21|   Mumbai|  West| Meera Patel|             UPI|   Delivered|                 286000|
|    1002| Ramesh Kumar|Printer|Electronics|      10|     12000| 120000|2023-08-05|    Delhi| North| Anil Sharma|  

DataFrame[category: string, total_revenue: bigint]

In [ ]:
#Silver: clean and enrich the data
df_silver=df_bronze \
  .dropDuplicates() \
  .dropna(subset=['order_id','product','revenue'])

df_silver = df_silver \
  .withColumn('order_date',to_date(col('order_date'),'dd-MM-yyyy')) \
  .withColumn('order_year',year(col('order_date'))) \
  .withColumn('order_month',month(col('order_date')))

df_silver=df_silver.withColumn('revenue_category',
                               F.when(col('revenue')>40000,'High')
                               .when(col('revenue')>10000,'Medium')
                               .otherwise('Low'))
print(f'Silver layer rows:{df_silver.count()}')
print("New columns added: order_date, order_year, order_month, revenue_category")
df_silver.select('order_date','order_year','order_month','revenue_category').show(5)
#df_silver.show(5,truncate=False)


Silver layer rows:5000
New columns added: order_date, order_year, order_month, revenue_category
+----------+----------+-----------+----------------+
|order_date|order_year|order_month|revenue_category|
+----------+----------+-----------+----------------+
|2023-11-08|      2023|         11|            High|
|2023-01-22|      2023|          1|          Medium|
|2023-09-29|      2023|          9|            High|
|2023-08-12|      2023|          8|            High|
|2023-11-09|      2023|         11|            High|
+----------+----------+-----------+----------------+
only showing top 5 rows


In [ ]:
#save silver layer as parquet
df_silver.write \
  .mode('overwrite') \
  .parquet('sales_silver.parquet')
print('silver parquet saved: sales_silver.parquet')
print(f'silver size:{get_dir_size("sales_silver.parquet"):.1f}kb')

df_verify=spark.read.parquet('sales_silver.parquet')
print(f'verify silver size:{get_dir_size("sales_silver.parquet"):.1f}kb')

#verify by reading it back
df_verify = spark.read.parquet('sales_silver.parquet')
print(f'Read-back-rowsL:{df_verify.count()}(should match silver count)')
df_verify.printSchema()


silver parquet saved: sales_silver.parquet
silver size:64.9kb
verify silver size:64.9kb
Read-back-rowsL:5000(should match silver count)
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- total_price_and_revenue: integer (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)



In [ ]:
# query 1 : top 5 products by total revenue

top_products=df_silver \
  .groupby('product') \
  .agg(\
      F.sum('revenue').alias('total_revenue'),\
      F.count('order_id').alias('num_order'),\
      spark_round(F.avg('revenue'), 2).alias('avg_revenue')\
  ) \
  .orderBy(F.col('total_revenue').asc()) \
  .limit(10)
top_products.show(truncate=False) #truncate used for alignment

+----------+-------------+---------+-----------+
|product   |total_revenue|num_order|avg_revenue|
+----------+-------------+---------+-----------+
|USB Hub   |2447400      |527      |4644.02    |
|Mouse     |3207200      |492      |6518.7     |
|Keyboard  |4878000      |495      |9854.55    |
|Webcam    |10982500     |532      |20643.8    |
|Headphones|13541500     |481      |28152.81   |
|Speaker   |16317000     |470      |34717.02   |
|Printer   |44544000     |488      |91278.69   |
|Monitor   |82126000     |481      |170740.12  |
|Tablet    |135104000    |532      |253954.89  |
|Laptop    |182700000    |502      |363944.22  |
+----------+-------------+---------+-----------+



In [ ]:
#

top_products=df_silver \
  .groupby('region') \
  .agg(\
      F.sum('revenue').alias('total_revenue'),\
      F.count('order_id').alias('num_order'),\
      F.countDistinct('customer_name').alias('Unique_customer') \
  ) \
  .orderBy(F.col('total_revenue').desc()) \
  .limit(5)
top_products.show(truncate=False) #truncate used for alignment

+------+-------------+---------+---------------+
|region|total_revenue|num_order|Unique_customer|
+------+-------------+---------+---------------+
|West  |198275600    |2021     |15             |
|South |147145900    |1483     |15             |
|North |99878400     |995      |15             |
|East  |50547700     |501      |15             |
+------+-------------+---------+---------------+



In [ ]:
#monthly_revenue_trend

monthly_revenue_trend = df_silver \
  .groupby('order_month') \
  .agg(\
      F.sum('revenue').alias('monthly_revenue'),\
      F.count('order_id').alias('monthly_orders')\
  ) \
  .orderBy('order_month')\
  .limit(12)

monthly_revenue_trend.show(truncate=False)

+-----------+---------------+--------------+
|order_month|monthly_revenue|monthly_orders|
+-----------+---------------+--------------+
|1          |41068200       |423           |
|2          |34485400       |375           |
|3          |40031200       |451           |
|4          |38857100       |390           |
|5          |39984500       |423           |
|6          |40707400       |390           |
|7          |42640700       |405           |
|8          |43718500       |418           |
|9          |37640200       |398           |
|10         |47839000       |479           |
|11         |44577100       |419           |
|12         |44298300       |429           |
+-----------+---------------+--------------+



In [43]:


monthly_revenue_trend = df_silver \
    .withColumn("month_name", F.date_format("order_date", "MMMM")) \
    .groupBy("month_name", "order_month") \
    .agg(
        F.sum("revenue").alias("monthly_revenue"),
        F.count("order_id").alias("monthly_orders")
    ) \
    .orderBy("order_month") \


monthly_revenue_trend.show(truncate=False)

+----------+-----------+---------------+--------------+
|month_name|order_month|monthly_revenue|monthly_orders|
+----------+-----------+---------------+--------------+
|January   |1          |41068200       |423           |
|February  |2          |34485400       |375           |
|March     |3          |40031200       |451           |
|April     |4          |38857100       |390           |
|May       |5          |39984500       |423           |
|June      |6          |40707400       |390           |
|July      |7          |42640700       |405           |
|August    |8          |43718500       |418           |
|September |9          |37640200       |398           |
|October   |10         |47839000       |479           |
|November  |11         |44577100       |419           |
|December  |12         |44298300       |429           |
+----------+-----------+---------------+--------------+

